# FAME — Final DRVM validation (V8)

This notebook validates the **same two-stage DRVM specification used in the audited manuscript**.

It has three goals:

1. verify finite-horizon false-signal behavior under the null;
2. quantify how detection power and delay vary with reference size \(m_{\rm ref}\), post-change horizon \(H\), and deterioration magnitude \(\Delta_R\);
3. compare empirical power with the manuscript's conservative first-order monitorability margin.

The monitored quantity is

\[
X_t=\min(R_t^{DR}/B,1).
\]

The reference mean is upper-bounded with a binary-KL bound at level \(1-\gamma\), and the sequential monitor uses the discrete \(\lambda\)-mixture and change-point weights

\[
\rho_k=\frac{1}{k(k+1)}
\]

specified in the manuscript.


In [ ]:
from __future__ import annotations

import math
import shutil
from dataclasses import dataclass, asdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
@dataclass(frozen=True)
class Config:
    nu: int = 251
    sigma_y: float = 0.25
    sigma_p: float = 0.50
    B: float = 5.0
    alpha: float = 0.04
    gamma: float = 0.01
    delta_regret: float = 0.01
    lambdas: tuple = (0.10, 0.25, 0.50, 1.00, 2.00, 4.00)
    seed: int = 20260908

    @property
    def delta_x(self):
        return self.delta_regret / self.B

cfg = Config()
cfg


## 1. Vectorized decision-representation regret

The simulation uses the same binary threshold representation family employed in the previous diagnostic studies. Candidate thresholds span \([-1,1]\), while the deployed representation uses threshold zero. The ex-post family oracle can therefore be evaluated exactly without a numerical grid search.

A nonzero regret occurs only when both actions are available within the frozen family and the deployed threshold chooses the action with lower realized utility.


In [ ]:
def regret_vector(yhat, y, kappa):
    yhat = np.asarray(yhat)
    y = np.asarray(y)
    kappa = np.asarray(kappa)

    # With thresholds in [-1,1], both binary actions are available iff -1 < yhat <= 1.
    both_actions = (yhat > -1.0) & (yhat <= 1.0)
    deployed_one = yhat > 0.0
    realized_gap = y - kappa

    r = np.zeros_like(y, dtype=float)

    # Deployed action 1, but action 0 is better.
    idx = both_actions & deployed_one & (realized_gap < 0)
    r[idx] = -realized_gap[idx]

    # Deployed action 0, but action 1 is better.
    idx = both_actions & (~deployed_one) & (realized_gap > 0)
    r[idx] = realized_gap[idx]

    return r


## 2. Reference upper bound and DRVM

The reference bound solves

\[
m\,\mathrm{kl}(\bar X_m,q)=\log(1/\gamma),
\qquad q\ge \bar X_m.
\]

The DRVM then uses

\[
c=r_0^U+\delta_X
\]

and the Bennett variance budget from the manuscript.


In [ ]:
def binary_kl(p, q):
    eps = 1e-15
    p = float(np.clip(p, eps, 1-eps))
    q = float(np.clip(q, eps, 1-eps))
    return p*math.log(p/q) + (1-p)*math.log((1-p)/(1-q))


def kl_upper_mean(xbar, m, gamma):
    xbar = float(np.clip(xbar, 0.0, 1.0))
    if xbar >= 1.0:
        return 1.0

    target = math.log(1/gamma)
    lo, hi = xbar, 1.0 - 1e-15

    for _ in range(100):
        mid = (lo + hi) / 2
        if m * binary_kl(xbar, mid) > target:
            hi = mid
        else:
            lo = mid

    return (lo + hi) / 2


LAMBDAS = np.asarray(cfg.lambdas, dtype=float)
PSI = np.exp(LAMBDAS) - 1.0 - LAMBDAS
WEIGHTS = np.repeat(1.0/len(LAMBDAS), len(LAMBDAS))


def drvm_monitor(x, c, cfg=cfg):
    v = c*(1-c) if c <= 0.5 else 0.25

    # Recursive implementation of the change-point mixture.
    A = np.zeros(len(LAMBDAS), dtype=float)
    threshold = 1.0 / cfg.alpha

    for idx, xt in enumerate(x):
        t = idx + 1
        rho_t = 1.0 / (t*(t+1.0))

        factor = np.exp(
            np.clip(
                LAMBDAS*(xt-c) - PSI*v,
                -745.0, 700.0
            )
        )

        A = factor * (A + rho_t)
        C_t = 1.0/(t+1.0) + A @ WEIGHTS

        if C_t >= threshold:
            return t

    return None


## 3. One simulation trajectory

In [ ]:
def simulate_reference(rng, m, cfg=cfg):
    latent = rng.normal(size=m)
    y = latent + rng.normal(0.0, cfg.sigma_y, m)
    yhat = latent + rng.normal(0.0, cfg.sigma_p, m)

    regret = regret_vector(yhat, y, np.zeros(m))
    x_ref = np.minimum(regret/cfg.B, 1.0)

    r0_upper = kl_upper_mean(
        xbar=x_ref.mean(),
        m=m,
        gamma=cfg.gamma
    )

    c = min(1.0, r0_upper + cfg.delta_x)

    return x_ref, regret, r0_upper, c


def simulate_deployment(rng, H, delta_R, cfg=cfg):
    T = cfg.nu + H - 1

    latent = rng.normal(size=T)
    y = latent + rng.normal(0.0, cfg.sigma_y, T)
    yhat = latent + rng.normal(0.0, cfg.sigma_p, T)

    kappa = np.zeros(T)
    kappa[cfg.nu-1:] = delta_R

    regret = regret_vector(yhat, y, kappa)
    x = np.minimum(regret/cfg.B, 1.0)

    return x, regret


def one_replication(seed, m_ref, H, delta_R, cfg=cfg):
    rng = np.random.default_rng(seed)

    x_ref, r_ref, r0_upper, c = simulate_reference(rng, m_ref, cfg)
    x, r = simulate_deployment(rng, H, delta_R, cfg)

    tau = drvm_monitor(x, c, cfg)

    post = np.arange(len(x)) >= cfg.nu-1

    return {
        "tau": np.nan if tau is None else tau,
        "early_signal": int(tau is not None and tau < cfg.nu),
        "post_detection": int(tau is not None and tau >= cfg.nu),
        "delay": np.nan if tau is None or tau < cfg.nu else tau-cfg.nu,
        "mean_x_ref": float(x_ref.mean()),
        "mean_x_post": float(x[post].mean()),
        "r0_upper": float(r0_upper),
        "c": float(c),
        "ref_clip_fraction": float(np.mean(r_ref > cfg.B)),
        "deploy_clip_fraction": float(np.mean(r > cfg.B)),
    }


## 4. Population moments for the planning boundary

The planning quantities are estimated from a large independent Monte Carlo sample, not from the power experiment.


In [ ]:
def population_moments(delta_values, n_large=1_000_000, seed=314159, cfg=cfg):
    rng = np.random.default_rng(seed)

    latent = rng.normal(size=n_large)
    y = latent + rng.normal(0.0, cfg.sigma_y, n_large)
    yhat = latent + rng.normal(0.0, cfg.sigma_p, n_large)

    rows = []

    for d in [0.0] + list(delta_values):
        regret = regret_vector(yhat, y, np.full(n_large, d))
        x = np.minimum(regret/cfg.B, 1.0)

        rows.append({
            "delta_R": d,
            "mu_x": float(x.mean()),
            "var_x": float(x.var()),
            "clip_fraction": float(np.mean(regret > cfg.B)),
        })

    return pd.DataFrame(rows)


DELTA_GRID = [1.00, 1.25, 1.50]
moments = population_moments(DELTA_GRID)
moments


## 5. Manuscript-aligned first-order monitorability boundary

For planning, we replace the realized reference error by the high-probability KL width evaluated at the population mean \(\mu_0\).

For the actual discrete mixture, component \(j\) has

\[
g_j(d,v)=\lambda_jd-v(e^{\lambda_j}-1-\lambda_j)
\]

and must overcome

\[
A_{\nu,j}
=
\log(1/\alpha)
+\log(1/\rho_\nu)
+\log(1/w_j).
\]

We solve the manuscript's discrete condition directly.


In [ ]:
MU0 = float(moments.loc[np.isclose(moments.delta_R, 0.0), "mu_x"].iloc[0])

def planning_reference_error(m_ref, cfg=cfg):
    return kl_upper_mean(MU0, m_ref, cfg.gamma) - MU0


def sequential_excess_discrete(m_ref, H, cfg=cfg):
    e_ref = planning_reference_error(m_ref, cfg)
    c = MU0 + cfg.delta_x + e_ref
    v = c*(1-c) if c <= 0.5 else 0.25

    rho_nu = 1.0 / (cfg.nu*(cfg.nu+1.0))
    w = 1.0 / len(LAMBDAS)

    A = (
        math.log(1.0/cfg.alpha)
        + math.log(1.0/rho_nu)
        + math.log(1.0/w)
    )

    def score(d):
        g = LAMBDAS*d - v*PSI
        return float(np.max(H*g - A))

    lo, hi = 0.0, 1.0
    while score(hi) < 0:
        hi *= 2.0

    for _ in range(120):
        mid = (lo+hi)/2
        if score(mid) >= 0:
            hi = mid
        else:
            lo = mid

    return e_ref, (lo+hi)/2, c, v


def planning_boundary(m_ref, H, delta_R, cfg=cfg):
    e_ref, e_seq, c, v = sequential_excess_discrete(m_ref, H, cfg)

    mu1 = float(
        moments.loc[
            np.isclose(moments.delta_R, delta_R),
            "mu_x"
        ].iloc[0]
    )

    delta_total = mu1 - MU0
    delta_min = cfg.delta_x + e_ref + e_seq

    return {
        "delta_total": delta_total,
        "e_ref": e_ref,
        "e_seq": e_seq,
        "delta_min": delta_min,
        "monitorability_margin": delta_total-delta_min,
        "planning_c": c,
        "planning_v": v,
    }


## 6. Power/delay experiment

The final grid uses:

\[
m_{\rm ref}\in\{250,500,1000,3000,5000\},
\]

\[
H\in\{125,250,500\},
\]

and

\[
\Delta_R\in\{1.00,1.25,1.50\}.
\]

Each cell uses 300 independent replications.


In [ ]:
M_GRID = [250, 500, 1000, 3000, 5000]
H_GRID = [125, 250, 500]
N_MC = 500

out_dir = Path("mc_results_v8")
out_dir.mkdir(parents=True, exist_ok=True)

total = len(M_GRID)*len(H_GRID)*len(DELTA_GRID)*N_MC
children = np.random.SeedSequence(cfg.seed + 8000).spawn(total)

rows = []
s = 0

for H in H_GRID:
    for m in M_GRID:
        for d in DELTA_GRID:
            print(f"H={H:3d}, m={m:4d}, delta_R={d:.2f}")

            boundary = planning_boundary(m, H, d, cfg)

            for rep in range(N_MC):
                seed = int(children[s].generate_state(1, dtype=np.uint64)[0])
                s += 1

                z = one_replication(seed, m, H, d, cfg)

                rows.append({
                    "H": H,
                    "m_ref": m,
                    "delta_R": d,
                    "rep": rep,
                    **z,
                    **boundary,
                })

raw_power = pd.DataFrame(rows)
raw_power.to_csv(out_dir/"power_raw_v8.csv", index=False)

summary_power = (
    raw_power.groupby(["H","m_ref","delta_R"], as_index=False)
    .agg(
        n=("rep","size"),
        p_early=("early_signal","mean"),
        p_detect=("post_detection","mean"),
        median_delay=("delay","median"),
        mean_x_ref=("mean_x_ref","mean"),
        mean_x_post=("mean_x_post","mean"),
        mean_r0_upper=("r0_upper","mean"),
        mean_c=("c","mean"),
        ref_clip_fraction=("ref_clip_fraction","mean"),
        deploy_clip_fraction=("deploy_clip_fraction","mean"),
        delta_total=("delta_total","first"),
        e_ref=("e_ref","first"),
        e_seq=("e_seq","first"),
        delta_min=("delta_min","first"),
        monitorability_margin=("monitorability_margin","first"),
    )
)

summary_power.to_csv(out_dir/"power_summary_v8.csv", index=False)
summary_power


## 7. Dedicated null experiment

The theorem controls the probability of *ever* issuing a false signal by \(\alpha+\gamma=0.05\). The finite simulation horizon is shorter than infinity, so the empirical false-signal rate is expected to be no larger than this bound and may be substantially smaller.

We evaluate the longest horizon (\(H=500\)) at both a short and a long reference sample.


In [ ]:
NULL_N = 5000
null_rows = []

for m in [250, 5000]:
    children = np.random.SeedSequence(cfg.seed + 9000 + m).spawn(NULL_N)

    for rep, child in enumerate(children):
        seed = int(child.generate_state(1, dtype=np.uint64)[0])
        z = one_replication(seed, m, 500, 0.0, cfg)

        null_rows.append({
            "m_ref": m,
            "H": 500,
            "rep": rep,
            **z,
        })

null_raw = pd.DataFrame(null_rows)
null_raw.to_csv(out_dir/"null_raw_v8.csv", index=False)

null_summary = (
    null_raw.groupby(["m_ref","H"], as_index=False)
    .agg(
        n=("rep","size"),
        p_false=("post_detection","mean"),
        p_early=("early_signal","mean"),
        p_any_signal=("tau", lambda x: x.notna().mean()),
        mean_c=("c","mean"),
        ref_clip_fraction=("ref_clip_fraction","mean"),
        deploy_clip_fraction=("deploy_clip_fraction","mean"),
    )
)

null_summary.to_csv(out_dir/"null_summary_v8.csv", index=False)
null_summary


## 8. Relation between the monitorability margin and power

In [ ]:
spearman_margin = (
    summary_power[["monitorability_margin","p_detect"]]
    .corr(method="spearman")
    .iloc[0,1]
)

print(f"Spearman correlation (monitorability margin vs power): {spearman_margin:.4f}")

margin_summary = summary_power[
    ["H","m_ref","delta_R","monitorability_margin","p_detect","median_delay"]
].sort_values("monitorability_margin")

margin_summary.to_csv(out_dir/"margin_power_relation_v8.csv", index=False)
margin_summary


## 9. Compact result table for the manuscript

In [ ]:
selected = summary_power[
    (
        ((summary_power.H == 125) & (summary_power.m_ref.isin([500,3000])) & (summary_power.delta_R == 1.50))
        |
        ((summary_power.H == 250) & (summary_power.m_ref.isin([500,1000,3000,5000])) & (summary_power.delta_R == 1.50))
        |
        ((summary_power.H == 500) & (summary_power.m_ref.isin([500,1000,3000,5000])) & (summary_power.delta_R.isin([1.00,1.25,1.50])))
    )
][
    ["H","m_ref","delta_R","p_detect","median_delay","monitorability_margin"]
].sort_values(["H","delta_R","m_ref"])

selected.to_csv(out_dir/"manuscript_selected_results_v8.csv", index=False)
selected


## 10. Figure: power versus conservative monitorability margin

In [ ]:
plt.figure(figsize=(8.5,5.2))

markers = {125:"o", 250:"s", 500:"^"}

for H in H_GRID:
    g = summary_power[summary_power.H == H]
    plt.scatter(
        g["monitorability_margin"],
        g["p_detect"],
        marker=markers[H],
        label=f"H={H}"
    )

plt.axvline(0, linestyle="--")
plt.xlabel("Conservative monitorability margin")
plt.ylabel("Empirical detection probability")
plt.ylim(-0.03, 1.03)
plt.legend()
plt.tight_layout()

fig_png = out_dir/"fig_drvm_margin_power_v8.png"
fig_pdf = out_dir/"fig_drvm_margin_power_v8.pdf"

plt.savefig(fig_png, dpi=220)
plt.savefig(fig_pdf)
plt.close()

print(fig_png)


## 11. Figure: reference size, horizon, and power at \(\Delta_R=1.50\)

In [ ]:
plt.figure(figsize=(8.5,5.2))

g = summary_power[np.isclose(summary_power.delta_R, 1.50)]

for H in H_GRID:
    gh = g[g.H == H].sort_values("m_ref")
    plt.plot(
        gh["m_ref"],
        gh["p_detect"],
        marker=markers[H],
        label=f"H={H}"
    )

plt.xlabel("Reference sample size")
plt.ylabel("Empirical detection probability")
plt.ylim(-0.03, 1.03)
plt.legend()
plt.tight_layout()

fig2_png = out_dir/"fig_drvm_reference_horizon_power_v8.png"
fig2_pdf = out_dir/"fig_drvm_reference_horizon_power_v8.pdf"

plt.savefig(fig2_png, dpi=220)
plt.savefig(fig2_pdf)
plt.close()

print(fig2_png)


## 12. Save configuration and archive

In [ ]:
pd.DataFrame([{
    **asdict(cfg),
    "m_grid": str(M_GRID),
    "H_grid": str(H_GRID),
    "delta_grid": str(DELTA_GRID),
    "n_mc_per_power_cell": N_MC,
    "n_mc_per_null_setting": NULL_N,
    "spearman_margin_power": spearman_margin,
}]).to_csv(out_dir/"config_v8.csv", index=False)

moments.to_csv(out_dir/"population_moments_v8.csv", index=False)

zip_path = shutil.make_archive(
    "mc_results_v8",
    "zip",
    root_dir=out_dir
)

print(f"Results archive: {zip_path}")


## Final stored run

The manuscript run used **500 replications per power cell** and **5,000 null replications at each of two reference sizes**. No null trajectory signalled over the finite horizon, and the Spearman correlation between the conservative monitorability margin and empirical power was **0.833**.

The complete machine-readable results are stored in `mc_results_v8.zip`.

In [ ]:
# Load the final stored outputs without rerunning the simulation
final_power = pd.read_csv('mc_results_v8/power_summary_v8.csv')
final_null = pd.read_csv('mc_results_v8/null_summary_v8.csv')
final_power.head(), final_null
